# 人工智能体构建第一次作业

基于 LangGraph 框架，实现质数求和计算与性能对比。

### 作业要求
1. 随机生成一个5维列表，元素均为大于1000且小于10000的正整数，并从小到大排列
2. 设计两种方法，给定正整数，计算小于该正整数的所有质数之和
3. 对列表中每个整数，用两种方法计算，通过自定义Python装饰器统计耗时，放大10000倍，形成两个新列表
4. 两个列表相减，并输出

## 架构设计

### 图结构
```
START → generate_numbers → compute_with_method_a → compute_with_method_b → calculate_difference → END
```

### 节点说明
| 节点 | 功能 |
|------|------|
| generate_numbers | 随机生成5维列表（1000~10000正整数），从小到大排列 |
| compute_with_method_a | 用逐一判断法计算质数之和，装饰器计时 |
| compute_with_method_b | 用埃拉托斯特尼筛法计算质数之和，装饰器计时 |
| calculate_difference | 两个耗时列表相减并输出 |

### 状态定义（AgentState）
| 字段 | 类型 | 说明 |
|------|------|------|
| numbers | list[int] | 生成的5维正整数列表 |
| results_a | list[float] | 方法A的耗时列表（放大10000倍） |
| results_b | list[float] | 方法B的耗时列表（放大10000倍） |
| difference | list[float] | 两个列表相减的结果 |

In [1]:
import random
import time
import functools
from typing import TypedDict
from langgraph.graph import StateGraph, END

print("依赖导入完成")

依赖导入完成


In [2]:
def timing_decorator(func):
    """自定义装饰器：统计函数执行时间，放大10000倍，返回 (结果, 放大后耗时)"""
    @functools.wraps(func)
    def wrapper(n):
        start = time.perf_counter()
        result = func(n)
        elapsed = (time.perf_counter() - start) * 10000
        return result, round(elapsed, 6)
    return wrapper

print("装饰器定义完成")

装饰器定义完成


In [3]:
@timing_decorator
def method_a_sum_primes(n):
    """方法A：逐一判断法求小于n的所有质数之和"""
    def is_prime(num):
        if num < 2:
            return False
        for i in range(2, int(num**0.5) + 1):
            if num % i == 0:
                return False
        return True
    return sum(i for i in range(2, n) if is_prime(i))


@timing_decorator
def method_b_sum_primes(n):
    """方法B：埃拉托斯特尼筛法求小于n的所有质数之和"""
    if n < 2:
        return 0
    sieve = [True] * n
    sieve[0] = sieve[1] = False
    for i in range(2, int(n**0.5) + 1):
        if sieve[i]:
            for j in range(i * i, n, i):
                sieve[j] = False
    return sum(i for i in range(2, n) if sieve[i])

print("两种质数求和算法定义完成")

两种质数求和算法定义完成


In [4]:
class AgentState(TypedDict):
    numbers: list
    results_a: list
    results_b: list
    difference: list

print("AgentState 定义完成")

AgentState 定义完成


In [5]:
def generate_numbers(state: AgentState) -> dict:
    """节点1：随机生成5个1000~10000的正整数，从小到大排列"""
    numbers = sorted(random.randint(1001, 9999) for _ in range(5))
    print(f"生成的数字列表: {numbers}")
    return {"numbers": numbers}


def compute_with_method_a(state: AgentState) -> dict:
    """节点2：用方法A（逐一判断法）计算每个数的质数之和并计时"""
    results_a = []
    for num in state["numbers"]:
        result, time_scaled = method_a_sum_primes(num)
        results_a.append(time_scaled)
        print(f"  方法A - n={num}: 质数之和={result}, 耗时×10000={time_scaled}")
    return {"results_a": results_a}


def compute_with_method_b(state: AgentState) -> dict:
    """节点3：用方法B（筛法）计算每个数的质数之和并计时"""
    results_b = []
    for num in state["numbers"]:
        result, time_scaled = method_b_sum_primes(num)
        results_b.append(time_scaled)
        print(f"  方法B - n={num}: 质数之和={result}, 耗时×10000={time_scaled}")
    return {"results_b": results_b}


def calculate_difference(state: AgentState) -> dict:
    """节点4：两个耗时列表相减"""
    diff = [round(a - b, 6) for a, b in zip(state["results_a"], state["results_b"])]
    print(f"\n方法A耗时列表: {state['results_a']}")
    print(f"方法B耗时列表: {state['results_b']}")
    print(f"相减结果(A-B): {diff}")
    return {"difference": diff}

print("4个节点函数定义完成")

4个节点函数定义完成


In [6]:
graph = StateGraph(AgentState)

graph.add_node("generate_numbers", generate_numbers)
graph.add_node("compute_with_method_a", compute_with_method_a)
graph.add_node("compute_with_method_b", compute_with_method_b)
graph.add_node("calculate_difference", calculate_difference)

graph.set_entry_point("generate_numbers")
graph.add_edge("generate_numbers", "compute_with_method_a")
graph.add_edge("compute_with_method_a", "compute_with_method_b")
graph.add_edge("compute_with_method_b", "calculate_difference")
graph.add_edge("calculate_difference", END)

app = graph.compile()

print("图构建完成！")
print(f"节点: {list(graph.nodes.keys())}")
print("图结构: START → generate_numbers → compute_with_method_a → compute_with_method_b → calculate_difference → END")

图构建完成！
节点: ['generate_numbers', 'compute_with_method_a', 'compute_with_method_b', 'calculate_difference']
图结构: START → generate_numbers → compute_with_method_a → compute_with_method_b → calculate_difference → END


In [7]:
result = app.invoke({"numbers": [], "results_a": [], "results_b": [], "difference": []})

print("\n" + "="*60)
print("最终结果")
print("="*60)
print(f"生成的数字列表: {result['numbers']}")
print(f"方法A耗时列表(×10000): {result['results_a']}")
print(f"方法B耗时列表(×10000): {result['results_b']}")
print(f"相减结果(A-B): {result['difference']}")

生成的数字列表: [2633, 3819, 4854, 7124, 9599]
  方法A - n=2633: 质数之和=456716, 耗时×10000=10.468
  方法A - n=3819: 质数之和=931686, 耗时×10000=15.914
  方法A - n=4854: 质数之和=1454367, 耗时×10000=21.046
  方法A - n=7124: 质数之和=3020151, 耗时×10000=32.468
  方法A - n=9599: 质数之和=5296125, 耗时×10000=46.642
  方法B - n=2633: 质数之和=456716, 耗时×10000=1.487
  方法B - n=3819: 质数之和=931686, 耗时×10000=2.019
  方法B - n=4854: 质数之和=1454367, 耗时×10000=2.653
  方法B - n=7124: 质数之和=3020151, 耗时×10000=3.827
  方法B - n=9599: 质数之和=5296125, 耗时×10000=5.386

方法A耗时列表: [10.468, 15.914, 21.046, 32.468, 46.642]
方法B耗时列表: [1.487, 2.019, 2.653, 3.827, 5.386]
相减结果(A-B): [8.981, 13.895, 18.393, 28.641, 41.256]

最终结果
生成的数字列表: [2633, 3819, 4854, 7124, 9599]
方法A耗时列表(×10000): [10.468, 15.914, 21.046, 32.468, 46.642]
方法B耗时列表(×10000): [1.487, 2.019, 2.653, 3.827, 5.386]
相减结果(A-B): [8.981, 13.895, 18.393, 28.641, 41.256]


## 结果分析

- **方法A（逐一判断法）**：对每个数逐一做质数判断，时间复杂度 O(n√n)，速度较慢
- **方法B（埃拉托斯特尼筛法）**：一次性筛出所有质数，时间复杂度 O(n log log n)，速度较快
- 相减结果 A-B 均为正数，说明方法A始终比方法B慢
- 耗时已放大10000倍，两种算法的性能差异在较大数字上更加明显
- 本项目使用 LangGraph 的 StateGraph 将四个步骤编排为线性流水线，通过自定义装饰器实现非侵入式计时